<a href="https://colab.research.google.com/github/AkibHasanGH/ml-assignment-colab/blob/main/ml_final_assignment.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [14]:
from google.colab import files
uploaded = files.upload()


Saving insurance.csv to insurance.csv


Data Preprocessing (10 Marks)

In [ ]:
from sklearn.preprocessing import StandardScaler

df = df.fillna(method="ffill")
df["sex"] = df["sex"].map({"male":0,"female":1})
df["smoker"] = df["smoker"].map({"yes":1,"no":0})

categorical_cols = df.select_dtypes(include=["object"]).columns
df = pd.get_dummies(df, columns=categorical_cols)

scaler = StandardScaler()
numeric_cols = ["age","bmi","children","charges"]
df[numeric_cols] = scaler.fit_transform(df[numeric_cols])

Q1 = df["charges"].quantile(0.25)
Q3 = df["charges"].quantile(0.75)
IQR = Q3 - Q1
df = df[(df["charges"] >= Q1 - 1.5*IQR) & (df["charges"] <= Q3 + 1.5*IQR)]




/tmp/ipykernel_948/3061240224.py:3: FutureWarning: DataFrame.fillna with 'method' is deprecated and will raise in a future version. Use obj.ffill() or obj.bfill() instead.
  df = df.fillna(method="ffill")


3. Pipeline Creation (10 Marks)

In [ ]:
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.linear_model import LinearRegression
from sklearn.preprocessing import OneHotEncoder

numeric_features = ["age","bmi","children"]
categorical_features = ["sex","smoker","region_northwest","region_southeast","region_southwest","region_northeast"]

numeric_transformer = StandardScaler()
categorical_transformer = OneHotEncoder(handle_unknown="ignore")

preprocessor = ColumnTransformer(
    transformers=[
        ("num", numeric_transformer, numeric_features),
        ("cat", categorical_transformer, categorical_features)
    ]
)

pipeline = Pipeline(steps=[("preprocessor", preprocessor),("model", LinearRegression())])


Model: LinearRegression

Justification: Insurance dataset‑এর target variable হলো charges, যা continuous numeric value। তাই regression algorithm উপযুক্ত। Linear Regression baseline হিসেবে ভালো কাজ করে এবং সহজে interpret করা যায়। চাইলে RandomForestRegressor বা GradientBoostingRegressor ব্যবহার করলে আরও ভালো accuracy পাওয়া যেতে পারে।

5. Model Training (10 Marks)


In [ ]:
from sklearn.model_selection import train_test_split

X = df.drop("charges", axis=1)
y = df["charges"]

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

pipeline.fit(X_train, y_train)


Pipeline(steps=[('preprocessor',
                 ColumnTransformer(transformers=[('num', StandardScaler(),
                                                  ['age', 'bmi', 'children']),
                                                 ('cat',
                                                  OneHotEncoder(handle_unknown='ignore'),
                                                  ['sex', 'smoker',
                                                   'region_northwest',
                                                   'region_southeast',
                                                   'region_southwest',
                                                   'region_northeast'])])),
                ('model', LinearRegression())])

🔹 6. Cross‑Validation (10 Marks)

In [ ]:
from sklearn.model_selection import cross_val_score
import numpy as np

scores = cross_val_score(pipeline, X_train, y_train, cv=5, scoring="r2")
print("CV Mean R²:", np.mean(scores))
print("CV Std Dev:", np.std(scores))


7. Hyperparameter Tuning (10 Marks)

In [ ]:
from sklearn.model_selection import GridSearchCV

param_grid = {
    "model__fit_intercept": [True, False],
    "model__positive": [True, False]
}

grid_search = GridSearchCV(pipeline, param_grid, cv=5, scoring="r2")
grid_search.fit(X_train, y_train)

print("Tested Parameters:", grid_search.cv_results_["params"])
print("Best Parameters:", grid_search.best_params_)
print("Best CV Score:", grid_search.best_score_)


8. Best Model Selection (10 Marks)

In [ ]:
best_model = grid_search.best_estimator_


9. Model Performance Evaluation (10 Marks)

In [ ]:
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
import numpy as np

y_pred = best_model.predict(X_test)

rmse = np.sqrt(mean_squared_error(y_test, y_pred))
mae = mean_absolute_error(y_test, y_pred)
r2 = r2_score(y_test, y_pred)

print("RMSE:", rmse)
print("MAE:", mae)
print("R²:", r2)


10. Web Interface with Gradio

In [ ]:
import gradio as gr
import pandas as pd
import joblib
import joblib

best_model = grid_search.best_estimator_
joblib.dump(best_model, "best_model.pkl")

import joblib

best_model = grid_search.best_estimator_
joblib.dump(best_model, "best_model.pkl")

best_model = joblib.load("best_model.pkl")

def predict_insurance(age, sex, bmi, children, smoker, region):
    input_data = pd.DataFrame({
        "age":[age],
        "sex":[sex],
        "bmi":[bmi],
        "children":[children],
        "smoker":[smoker],
        "region":[region]
    })
    prediction = best_model.predict(input_data)[0]
    return f"Predicted Insurance Charges: {prediction:.2f}"

interface = gr.Interface(
    fn=predict_insurance,
    inputs=[
        gr.Number(label="Age"),
        gr.Radio(["male","female"], label="Sex"),
        gr.Number(label="BMI"),
        gr.Number(label="Children"),
        gr.Radio(["yes","no"], label="Smoker"),
        gr.Dropdown(["northwest","northeast","southeast","southwest"], label="Region")
    ],
    outputs="text",
    title="Insurance Charges Prediction"
)

interface.launch()


11. Deployment to Hugging Face

In [ ]:
import gradio as gr
import pandas as pd
import joblib

best_model = joblib.load("best_model.pkl")

def predict_insurance(age, sex, bmi, children, smoker, region):
    input_data = pd.DataFrame({
        "age":[age],
        "sex":[sex],
        "bmi":[bmi],
        "children":[children],
        "smoker":[smoker],
        "region":[region]
    })
    prediction = best_model.predict(input_data)[0]
    return f"Predicted Insurance Charges: {prediction:.2f}"

interface = gr.Interface(
    fn=predict_insurance,
    inputs=[
        gr.Number(label="Age"),
        gr.Radio(["male","female"], label="Sex"),
        gr.Number(label="BMI"),
        gr.Number(label="Children"),
        gr.Radio(["yes","no"], label="Smoker"),
        gr.Dropdown(["northwest","northeast","southeast","southwest"], label="Region")
    ],
    outputs="text",
    title="Insurance Charges Prediction"
)

interface.launch()
